# Performance- und Skalierungstests

Untersuchung der Rechenzeit des Modells auf `dataset/multimodal_network.json`

1. **Instanz 1: Schnell** (~ 1 Minute)
2. **Instanz 2: Medium** ()
3. **Instanz 3: Langsam** ()

---

## 1. Setup und Imports

In [ ]:
import sys
import time
import json
from pathlib import Path
from collections import defaultdict

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from freight_routing.data_loader import NetworkDataLoader
from freight_routing.data_models import Shipment, ObjectiveWeights, TransportArcTemplate
from freight_routing.model import TimeExpandedFreightRoutingModel

## 2. Laden der Netzwerkdaten

In [2]:
network_path = PROJECT_ROOT / "dataset/multimodal_network.json"

print(f"Lese Netzwerkdatei von: {network_path}...")
network_data = NetworkDataLoader.from_json(network_path)

network_data.summary()

Lese Netzwerkdatei von: /home/benedikt/Projects/Sustainable_Freight_Mode_Choice/dataset/multimodal_network.json...
Summary NetworkData:
hubs=870
arcs=36272
modes=4


## 3. Hilfsfunktionen zur Sendungsgenerierung und Durchführung der Tests

In [ ]:
def generate_shipments(network_data, shipment_count, planning_days):
    grouped_arcs = defaultdict(list)
    for arc in network_data.arc_templates:
        if isinstance(arc, TransportArcTemplate):
            grouped_arcs[(arc.from_hub, arc.to_hub)].append(arc)

    candidate_pairs = [
        (pair, arcs)
        for pair, arcs in grouped_arcs.items()
        if any(0 in arc.departure_minutes for arc in arcs)
    ]
    candidate_pairs.sort(
        key=lambda item: (-len({arc.mode for arc in item[1]}), item[0][0], item[0][1])
    )

    deadline = planning_days * 24 * 60
    shipments = []
    for index in range(shipment_count):
        (start_hub, end_hub), _ = candidate_pairs[index % len(candidate_pairs)]
        weight = 0.5 + (index % 5) * 0.25
        shipments.append(
            Shipment(
                id=f"scaling_shipment_{index + 1}",
                start_hub=start_hub,
                end_hub=end_hub,
                start_time=0,
                deadline=deadline,
                max_price=1_000_000.0,
                max_emissions=1_000_000.0,
                weight=weight,
            )
        )
    return shipments


def run_scaling_test(planning_days, shipment_count, time_limit_sec=None):
    print(f"\n{'='*60}")
    print(
        f"Starte Instanz: {planning_days} Tage Planungshorizont | {shipment_count} Sendung(en)"
    )
    print(f"{'='*60}")

    # Sendungen erzeugen
    shipments = generate_shipments(network_data, shipment_count, planning_days)
    print(f"-> Generiert: {len(shipments)} Sendung(en)")
    for s in shipments:
        print(
            f"   - {s.id}: {s.start_hub} -> {s.end_hub} | Gewicht: {s.weight}t | Deadline: {s.deadline} min"
        )

    model = TimeExpandedFreightRoutingModel(network_data)
    t_build_start = time.time()
    model.build(planning_days=planning_days, shipments=shipments)
    t_build = time.time() - t_build_start

    print(f"-> Modellaufbau abgeschlossen in {t_build:.2f} Sekunden.")
    model.summary()

    print(f"-> Löse Optimierungsproblem (Zeitlimit: {time_limit_sec}s)... ")
    t_solve_start = time.time()
    result = model.solve(shipments, time_limit_sec=time_limit_sec)
    t_solve = time.time() - t_solve_start

    print(f"-> Lösungsvorgang abgeschlossen in {t_solve:.2f} Sekunden.")
    print(f"   - Status: {result.status}")
    print(f"   - Optimalität erreicht: {result.is_optimal}")
    if result.is_optimal:
        print(f"   - Gesamtkosten: {result.total_cost:.2f} EUR")
        print(f"   - Gesamtemissionen: {result.total_emissions:.2f} kg CO2")
    return t_build, t_solve, result

---

## 4. Instanz 1: Schnell (Lösungszeit < 5 Minuten)

In [4]:
run_scaling_test(planning_days=2, shipment_count=1)


Starte Instanz: 2 Tage Planungshorizont | 1 Sendung(en)
-> Generiert: 1 Sendung(en)
   - scaling_shipment_1: SAN_297 -> LOS_284 | Gewicht: 0.5t | Deadline: 2880 min
-> Modellaufbau abgeschlossen in 1.59 Sekunden.
Summary TimeExpandedFreightRoutingModel:
planning_days=2
planning_horizon_min=2880
nodes=184112
arcs=326818
  - transport_arcs=124788
  - transfer_arcs=20048
  - waiting_arcs=181982
-> Löse Optimierungsproblem (Zeitlimit: Nones)... 
-> Lösungsvorgang abgeschlossen in 58.22 Sekunden.
   - Status: Optimal
   - Optimalität erreicht: True
   - Gesamtkosten: 517.74 EUR
   - Gesamtemissionen: 57.58 kg CO2


(1.5897605419158936,
 58.22461152076721,
 RoutingResult(status='Optimal', is_optimal=True, objective_value=0.03981644332881472, total_cost=517.74, total_emissions=57.5805, total_time=426.0, shipment_routes={'scaling_shipment_1': (_TimedArc(from_node=NetworkNode(hub_id='SAN_297', mode='road', time_min=0), to_node=NetworkNode(hub_id='LOS_284', mode='road', time_min=426), mode='road', arc_type=<ArcType.TRANSPORT: 'transport'>, departure_min=0, arrival_min=426, cost=735.4799999999999, emissions=55.160999999999994, capacity=10.0, distance=612.9, max_vehicles=None, fixed_cost=None, fixed_emissions=None),)}, total_fixed_cost=150.0, total_variable_cost=367.73999999999995, total_fixed_emissions=30.0, total_variable_emissions=27.580499999999997, diagnostics=()))

## 5. Instanz 2: Medium (Lösungszeit ~5 Minuten)

In [5]:
run_scaling_test(planning_days=2, shipment_count=5)


Starte Instanz: 2 Tage Planungshorizont | 5 Sendung(en)
-> Generiert: 5 Sendung(en)
   - scaling_shipment_1: SAN_297 -> LOS_284 | Gewicht: 0.5t | Deadline: 2880 min
   - scaling_shipment_2: BER_3970 -> LEI_3981 | Gewicht: 0.75t | Deadline: 2880 min
   - scaling_shipment_3: BER_3970 -> SZC_8009 | Gewicht: 1.0t | Deadline: 2880 min
   - scaling_shipment_4: BRE_3979 -> HAM_3971 | Gewicht: 1.25t | Deadline: 2880 min
   - scaling_shipment_5: CHI_285 -> IND_295 | Gewicht: 1.5t | Deadline: 2880 min
-> Modellaufbau abgeschlossen in 1.41 Sekunden.
Summary TimeExpandedFreightRoutingModel:
planning_days=2
planning_horizon_min=2880
nodes=184112
arcs=326818
  - transport_arcs=124788
  - transfer_arcs=20048
  - waiting_arcs=181982
-> Löse Optimierungsproblem (Zeitlimit: Nones)... 
-> Lösungsvorgang abgeschlossen in 307.69 Sekunden.
   - Status: Optimal
   - Optimalität erreicht: True
   - Gesamtkosten: 2172.33 EUR
   - Gesamtemissionen: 256.67 kg CO2


(1.4147684574127197,
 307.6904664039612,
 RoutingResult(status='Optimal', is_optimal=True, objective_value=0.09655009600886671, total_cost=2172.33, total_emissions=256.67475, total_time=959.0, shipment_routes={'scaling_shipment_1': (_TimedArc(from_node=NetworkNode(hub_id='SAN_297', mode='road', time_min=0), to_node=NetworkNode(hub_id='LOS_284', mode='road', time_min=426), mode='road', arc_type=<ArcType.TRANSPORT: 'transport'>, departure_min=0, arrival_min=426, cost=735.4799999999999, emissions=55.160999999999994, capacity=10.0, distance=612.9, max_vehicles=None, fixed_cost=None, fixed_emissions=None),), 'scaling_shipment_2': (_TimedArc(from_node=NetworkNode(hub_id='BER_3970', mode='road', time_min=0), to_node=NetworkNode(hub_id='LEI_3981', mode='road', time_min=129), mode='road', arc_type=<ArcType.TRANSPORT: 'transport'>, departure_min=0, arrival_min=129, cost=227.04, emissions=17.028, capacity=10.0, distance=189.2, max_vehicles=None, fixed_cost=None, fixed_emissions=None),), 'scaling_

## 6. Instanz 3: ~10 Min

In [6]:
run_scaling_test(planning_days=3, shipment_count=5)


Starte Instanz: 3 Tage Planungshorizont | 5 Sendung(en)
-> Generiert: 5 Sendung(en)
   - scaling_shipment_1: SAN_297 -> LOS_284 | Gewicht: 0.5t | Deadline: 4320 min
   - scaling_shipment_2: BER_3970 -> LEI_3981 | Gewicht: 0.75t | Deadline: 4320 min
   - scaling_shipment_3: BER_3970 -> SZC_8009 | Gewicht: 1.0t | Deadline: 4320 min
   - scaling_shipment_4: BRE_3979 -> HAM_3971 | Gewicht: 1.25t | Deadline: 4320 min
   - scaling_shipment_5: CHI_285 -> IND_295 | Gewicht: 1.5t | Deadline: 4320 min
-> Modellaufbau abgeschlossen in 2.15 Sekunden.
Summary TimeExpandedFreightRoutingModel:
planning_days=3
planning_horizon_min=4320
nodes=279907
arcs=499862
  - transport_arcs=191303
  - transfer_arcs=30782
  - waiting_arcs=277777
-> Löse Optimierungsproblem (Zeitlimit: Nones)... 
-> Lösungsvorgang abgeschlossen in 613.35 Sekunden.
   - Status: Optimal
   - Optimalität erreicht: True
   - Gesamtkosten: 2172.33 EUR
   - Gesamtemissionen: 256.67 kg CO2


(2.1504719257354736,
 613.3516900539398,
 RoutingResult(status='Optimal', is_optimal=True, objective_value=0.0639765293038714, total_cost=2172.33, total_emissions=256.67475, total_time=959.0, shipment_routes={'scaling_shipment_1': (_TimedArc(from_node=NetworkNode(hub_id='SAN_297', mode='road', time_min=0), to_node=NetworkNode(hub_id='LOS_284', mode='road', time_min=426), mode='road', arc_type=<ArcType.TRANSPORT: 'transport'>, departure_min=0, arrival_min=426, cost=735.4799999999999, emissions=55.160999999999994, capacity=10.0, distance=612.9, max_vehicles=None, fixed_cost=None, fixed_emissions=None),), 'scaling_shipment_2': (_TimedArc(from_node=NetworkNode(hub_id='BER_3970', mode='road', time_min=0), to_node=NetworkNode(hub_id='LEI_3981', mode='road', time_min=129), mode='road', arc_type=<ArcType.TRANSPORT: 'transport'>, departure_min=0, arrival_min=129, cost=227.04, emissions=17.028, capacity=10.0, distance=189.2, max_vehicles=None, fixed_cost=None, fixed_emissions=None),), 'scaling_s

## 7. Instanz 3: ~30 Min

In [ ]:
run_scaling_test(planning_days=5, shipment_count=5)


Starte Instanz: 5 Tage Planungshorizont | 5 Sendung(en)
-> Generiert: 5 Sendung(en)
   - scaling_shipment_1: SAN_297 -> LOS_284 | Gewicht: 0.5t | Deadline: 7200 min
   - scaling_shipment_2: BER_3970 -> LEI_3981 | Gewicht: 0.75t | Deadline: 7200 min
   - scaling_shipment_3: BER_3970 -> SZC_8009 | Gewicht: 1.0t | Deadline: 7200 min
   - scaling_shipment_4: BRE_3979 -> HAM_3971 | Gewicht: 1.25t | Deadline: 7200 min
   - scaling_shipment_5: CHI_285 -> IND_295 | Gewicht: 1.5t | Deadline: 7200 min
-> Modellaufbau abgeschlossen in 5.19 Sekunden.
Summary TimeExpandedFreightRoutingModel:
planning_days=5
planning_horizon_min=7200
nodes=474155
arcs=851334
  - transport_arcs=327059
  - transfer_arcs=52250
  - waiting_arcs=472025
-> Löse Optimierungsproblem (Zeitlimit: Nones)... 
-> Lösungsvorgang abgeschlossen in 5047.75 Sekunden.
   - Status: Optimal
   - Optimalität erreicht: True
   - Gesamtkosten: 2172.33 EUR
   - Gesamtemissionen: 256.67 kg CO2


(5.1927809715271,
 5047.747670412064,
 RoutingResult(status='Optimal', is_optimal=True, objective_value=0.03820066148082049, total_cost=2172.33, total_emissions=256.67475, total_time=959.0, shipment_routes={'scaling_shipment_1': (_TimedArc(from_node=NetworkNode(hub_id='SAN_297', mode='road', time_min=0), to_node=NetworkNode(hub_id='LOS_284', mode='road', time_min=426), mode='road', arc_type=<ArcType.TRANSPORT: 'transport'>, departure_min=0, arrival_min=426, cost=735.4799999999999, emissions=55.160999999999994, capacity=10.0, distance=612.9, max_vehicles=None, fixed_cost=None, fixed_emissions=None),), 'scaling_shipment_2': (_TimedArc(from_node=NetworkNode(hub_id='BER_3970', mode='road', time_min=0), to_node=NetworkNode(hub_id='LEI_3981', mode='road', time_min=129), mode='road', arc_type=<ArcType.TRANSPORT: 'transport'>, departure_min=0, arrival_min=129, cost=227.04, emissions=17.028, capacity=10.0, distance=189.2, max_vehicles=None, fixed_cost=None, fixed_emissions=None),), 'scaling_shi